# 12 — Business Interpretation and Recommendations

**Production note:** This notebook reads the validated production outputs from `scripts/` and `data/outputs/`. It does not re-fit models or re-run model selection. All metrics and forecast values shown by code cells are loaded from the existing CSV outputs, so the notebook does not hand-enter model results.

Scope:

- Restate the selected model result for each target region.
- Show training history, 2025 validation actuals, selected 2025 prediction, and selected 2026-2027 forecast for each target.
- Interpret model limitations, feature meaning, and business implications for Repsol.
- Identify internal Repsol data that would make the modeling stronger.

## 0. Setup — Load Validated Production Outputs

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import subprocess
import sys
from IPython.display import display, Markdown

cwd = Path().resolve()
REPO_ROOT = cwd if (cwd / 'data').exists() and (cwd / 'notebooks').exists() else cwd.parent
DATA_FEATURES = REPO_ROOT / 'data' / 'features'
DATA_OUTPUTS = REPO_ROOT / 'data' / 'outputs'
FIGS = REPO_ROOT / 'reports' / 'figures'
FIGS.mkdir(parents=True, exist_ok=True)

TARGETS = ['Nacional', 'Madrid', 'Cataluña', 'Andalucía', 'Valencia']
TARGET_COLORS = {
    'Nacional': '#FF6B35',
    'Madrid': '#004E89',
    'Cataluña': '#1A936F',
    'Andalucía': '#C84B31',
    'Valencia': '#8E44AD',
}
HEADLINE_FINAL_MODELS = {
    'SARIMA', 'SARIMAX', 'Logistic', 'Gompertz',
    'Ridge', 'Random Forest', 'XGBoost'
}

required_files = {
    'features': DATA_FEATURES / 'features_modelo_completo.csv',
    'final_metrics': DATA_OUTPUTS / 'metricas_final_selected.csv',
    'all_metrics': DATA_OUTPUTS / 'metricas_models.csv',
    'predictions': DATA_OUTPUTS / 'predicciones_test_2025.csv',
    'forecasts': DATA_OUTPUTS / 'forecast_24m_sarima_rf_xgb.csv',
    'selected_forecasts': DATA_OUTPUTS / 'forecast_24m_selected.csv',
    'acceptance': DATA_OUTPUTS / 'phase2_model_acceptance.csv',
    'pooling': DATA_OUTPUTS / 'phase2_pooling_decision.csv',
    'sarima_grid': DATA_OUTPUTS / 'sarima_grid_search_results.csv',
    'sarima_acceptance': DATA_OUTPUTS / 'sarima_order_acceptance.csv',
    'walkforward': DATA_OUTPUTS / 'model_selection_walkforward.csv',
    'sarima_drivers': DATA_OUTPUTS / 'selected_model_sarima_drivers.csv',
    'curve_params': DATA_OUTPUTS / 'selected_model_curve_parameters.csv',
    'curve_seasonal': DATA_OUTPUTS / 'selected_model_curve_seasonal.csv',
    'mini_check': DATA_OUTPUTS / 'mini_model_cross_check.csv',
    'sarima_ci': DATA_OUTPUTS / 'forecast_24m_sarima_confidence_intervals.csv',
}
missing = [str(path.relative_to(REPO_ROOT)) for path in required_files.values() if not path.exists()]
if missing:
    raise FileNotFoundError(
        f'Missing required production output files: {missing}. '
        "If the 'selected_model_*' driver files or 'mini_model_cross_check.csv' are the ones "
        'missing, run notebooks/11_mini_trend_regulation_model.ipynb then '
        'scripts/07_selected_model_drivers.py (after scripts/05 and scripts/06).'
    )

def load_outputs():
    out = {name: pd.read_csv(path) for name, path in required_files.items()}
    for df in [out['features'], out['predictions'], out['forecasts'], out['selected_forecasts']]:
        df['Fecha_Date'] = pd.to_datetime(df['Fecha'])
    return out

loaded = load_outputs()
features = loaded['features']
final_metrics = loaded['final_metrics']
all_metrics = loaded['all_metrics']
predictions = loaded['predictions']
forecasts = loaded['forecasts']
selected_forecasts = loaded['selected_forecasts']
acceptance = loaded['acceptance']
pooling = loaded['pooling']
sarima_grid = loaded['sarima_grid']
sarima_acceptance = loaded['sarima_acceptance']
walkforward = loaded['walkforward']
sarima_drivers = loaded['sarima_drivers']
curve_params = loaded['curve_params']
curve_seasonal = loaded['curve_seasonal']
mini_check = loaded['mini_check']
sarima_ci = loaded['sarima_ci']

selected_models = dict(zip(final_metrics['Target'], final_metrics['Model']))
if set(selected_models.values()) - HEADLINE_FINAL_MODELS:
    print('Detected stale non-headline-eligible selected outputs. Rebuilding production modeling outputs...')
    subprocess.run([sys.executable, str(REPO_ROOT / 'scripts' / '05_modeling_with_cnmc.py')], check=True, cwd=REPO_ROOT)
    loaded = load_outputs()
    features = loaded['features']
    final_metrics = loaded['final_metrics']
    all_metrics = loaded['all_metrics']
    predictions = loaded['predictions']
    forecasts = loaded['forecasts']
    selected_forecasts = loaded['selected_forecasts']
    acceptance = loaded['acceptance']
    pooling = loaded['pooling']
    sarima_grid = loaded['sarima_grid']
    sarima_acceptance = loaded['sarima_acceptance']
    walkforward = loaded['walkforward']
    sarima_drivers = loaded['sarima_drivers']
    curve_params = loaded['curve_params']
    curve_seasonal = loaded['curve_seasonal']
    mini_check = loaded['mini_check']
    sarima_ci = loaded['sarima_ci']
    selected_models = dict(zip(final_metrics['Target'], final_metrics['Model']))
    if set(selected_models.values()) - HEADLINE_FINAL_MODELS:
        raise ValueError(f'Final selected models are not all headline-eligible after rebuild: {selected_models}')

# The driver-detail files (selected_model_sarima_drivers.csv, selected_model_curve_parameters.csv,
# mini_model_cross_check.csv) are produced by a separate, fast script (scripts/07_selected_model_drivers.py)
# that is not wired into scripts/05's rebuild above, so they can silently go stale if the headline
# selection changes (e.g. a future data refresh) without that script being re-run. Fail loudly rather
# than display driver detail for the wrong model.
sarima_targets_now = set(final_metrics.loc[final_metrics['Model'].eq('SARIMA'), 'Target'])
curve_targets_now = set(final_metrics.loc[final_metrics['Model'].isin(['Logistic', 'Gompertz']), 'Target'])
sarima_drivers_targets = set(sarima_drivers['Target'].unique()) if not sarima_drivers.empty else set()
curve_params_targets = set(curve_params['Target'].unique()) if not curve_params.empty else set()
if sarima_drivers_targets != sarima_targets_now or curve_params_targets != curve_targets_now:
    raise ValueError(
        'selected_model_sarima_drivers.csv / selected_model_curve_parameters.csv are stale relative to '
        f'the current selection (SARIMA targets now: {sarima_targets_now}, found: {sarima_drivers_targets}; '
        f'curve targets now: {curve_targets_now}, found: {curve_params_targets}). '
        're-run scripts/07_selected_model_drivers.py.'
    )
if set(mini_check['Production_Model']) != set(selected_models.values()) or len(mini_check) != len(TARGETS):
    raise ValueError(
        'mini_model_cross_check.csv looks stale relative to the current selection. '
        're-run scripts/07_selected_model_drivers.py.'
    )

print('Loaded production outputs from:', REPO_ROOT)
print('Final selected models:', selected_models)


## 1. What the Models Can and Cannot Tell Us

The current project estimates **total market demand for biodiesel sold/reported as its own distinct product line** (the CORES/CNMC `BIODIESEL` category) for the national total and four selected regions. It does **not** estimate Repsol's own sales, station-level volumes, margins, or logistics needs, **and it does not estimate the biodiesel blended at low concentration into ordinary Gasóleo A diesel under the national mandate** — that volume is reported separately by CNMC under `GASÓLEO A` and is outside this project's target. This scope was confirmed directly with the Repsol representative; see `README.md`'s "Target Definition" section and `notebooks/11_mini_trend_regulation_model.ipynb` for the full technical distinction.

What the current models can support:

- Directional 2026-2027 market-demand planning by target region.
- Comparison of seven headline candidates with 2025 used only as an honest holdout report.
- Identification of where engineered variables help versus where history-only baselines still look stronger.
- Discussion of which additional Repsol internal data would make the forecast operationally stronger.

What the current models cannot prove alone:

- Repsol market share by region.
- Station-level product demand.
- Inventory or logistics requirements.
- Price elasticity specific to Repsol customers.
- A clean causal mandate effect from only 2023-2025 history.

### 1.1 Exact Selected Results by Region

The table below is loaded directly from `data/outputs/metricas_final_selected.csv`. These are the final selected 2025 holdout metrics.

In [ ]:
display(final_metrics.sort_values('Target').reset_index(drop=True))


### 1.2 Exact Feature-Aware Selection Decisions

The table below is loaded directly from `data/outputs/phase2_model_acceptance.csv`. It shows the final eligibility rule, the training-window walk-forward proposal, and the training-only selected winner and reported 2025 holdout metric.


In [ ]:
display(acceptance.sort_values('Target').reset_index(drop=True))

### 1.3 SARIMA Grid-Search Robustness Check

SARIMA parameter tuning ranking is training-only: every candidate order is scored and filtered for degeneracy using only the 2023-2024 training window (the latter check is also what excludes SARIMAX as a headline candidate for 4 of 5 targets; see `data/outputs/degenerate_fits.csv`). The single training-only winner per target then goes through one additional, disclosed step: a post-hoc shippability safety check fits that winner on the full 2023-2025 history (the same data the production forecast actually ships with) and verifies the resulting 24-month forecast is not degenerate. This is the only place 2025 data is used anywhere in SARIMA order selection, and it can only veto the single winner -- it never ranks or filters the candidate grid.

If the winner fails that safety check, the pipeline does not auto-substitute another candidate (which would just reintroduce a 2025-touching selection one step removed) or silently fall back to the plain SARIMA default order (which can score far worse on the only leak-free metric available). It requires an explicit, reviewed entry in `SARIMA_SAFETY_OVERRIDES` (`scripts/05_modeling_with_cnmc.py`), or it raises. **Cataluña's training-only winner currently fails this check and uses such a reviewed override** -- see the `Safety_Check_Degenerate` / `Override_Applied` columns below and `data/outputs/sarima_safety_check.csv`; the resulting order, (0,1,2)(1,0,0,12), is unchanged from before this disclosure was added.

In [ ]:
display(Markdown('#### Training-only SARIMA grid winners'))
display(
    sarima_grid[sarima_grid['Selected']]
    [['Target', 'p', 'd', 'q', 'P', 'D', 'Q', 'm', 'WalkForward_MAPE', 'Successful_Folds']]
    .sort_values('Target')
    .reset_index(drop=True)
)

display(Markdown('#### SARIMA production-order acceptance'))
display(
    sarima_acceptance[
        [
            'Target', 'Default_Order', 'Default_Seasonal_Order',
            'Grid_Selected_Order', 'Grid_Selected_Seasonal_Order',
            'Grid_WalkForward_MAPE', 'Safety_Check_Degenerate', 'Override_Applied',
            'Selected_By_Training_WalkForward',
            'Production_Order', 'Production_Seasonal_Order', 'Decision'
        ]
    ]
    .sort_values('Target')
    .reset_index(drop=True)
)

### 1.3 Candidate Model Metrics by Region

The following tables sort every tested candidate by validation MAPE within each target. These tables are loaded directly from `data/outputs/metricas_models.csv`.

In [ ]:
for target in TARGETS:
    display(Markdown(f'#### {target}'))
    cols = ['Target', 'Model', 'MAE', 'RMSE', 'MAPE', 'R2']
    display(
        all_metrics.loc[all_metrics['Target'].eq(target), cols]
        .sort_values(['MAPE', 'Model'])
        .reset_index(drop=True)
    )


### 1.4 Regional Training, Validation, and Forecast Plots

Each plot is generated from the production feature, prediction, and forecast CSVs:

- Training actuals: 2023-01 to 2024-12.
- 2025 validation actuals.
- 2025 prediction from the final selected model.
- 2026-2027 forecast from the final selected model.
- An uncertainty band around the forecast, built the same way as `scripts/05_modeling_with_cnmc.py`'s own chart: for Cataluña and Andalucía (SARIMA-selected), this is a real calibrated prediction interval from the fit's own forecast-error variance; for Nacional, Madrid, and Valencia (Logistic/Gompertz-selected), no native calibrated interval exists for a curve-fit model without bootstrapping, so the band shown is an illustrative proxy scaled off the 2025 holdout error, explicitly not a statistical interval. The two kinds of band are not directly comparable -- see the legend on each chart.

  **The calibrated band is shown at 50%, not the textbook-default 95%, and this is a deliberate display choice, not a smaller estimate of the true uncertainty.** At a 24-month horizon, a log1p-fit SARIMA model's 95% interval explodes asymmetrically once back-transformed (Cataluña: a ~3,400 Tm point forecast against a ~232,000 Tm *95%* upper bound by month 24) -- mathematically honest, but unreadable on a chart and not a usable planning number. The 50% interval shown here is the same calibrated method, just a less conservative, legible level; the full 95% figure is still on the record (see `memory.md` and `README.md`'s "Audit Fixes" history) and remains available from `predict_sarima_with_ci(result, n_steps, alpha=0.05)` if ever needed for a more conservative analysis.

The generated image files are saved under `reports/figures/` so they can be reused in slides or the final report.

In [ ]:
def safe_target_name(target: str) -> str:
    replacements = {
        'Nacional': 'national',
        'Madrid': 'madrid',
        'Cataluña': 'cataluna',
        'Andalucía': 'andalucia',
        'Valencia': 'valencia',
    }
    return replacements[target]

# Same calibrated-vs-heuristic uncertainty band logic as scripts/05_modeling_with_cnmc.py's
# plot_outputs(): SARIMA-selected targets get a real prediction interval from the fit's own
# forecast-error variance (predict_sarima_with_ci); Logistic/Gompertz-selected targets get an
# illustrative band scaled off the 2025 holdout error, since no native calibrated interval
# exists for curve-fit models without bootstrapping.
#
# SARIMA_CHART_CI_ALPHA must match scripts/05_modeling_with_cnmc.py's constant of the same
# name -- that script produced data/outputs/forecast_24m_sarima_confidence_intervals.csv at
# this alpha, so the label here has to describe the same level, not the textbook-default 95%.
# 0.5 (50%) is a deliberate display choice: the true 95% interval explodes asymmetrically by
# month 24 after back-transforming out of log1p space (e.g. Cataluña: ~3,400 Tm point forecast
# against a ~232,000 Tm 95% upper bound), which is mathematically honest but unreadable on a
# chart and not a usable planning number. 50% is not a claim that the real uncertainty is
# smaller -- it is the same calibrated interval at a less conservative, more legible level.
SARIMA_CHART_CI_ALPHA = 0.5
ci_level_pct = round((1 - SARIMA_CHART_CI_ALPHA) * 100)
selected_error = final_metrics.set_index('Target')[['MAPE', 'RMSE']].to_dict('index')

figure_rows = []
for target in TARGETS:
    model = selected_models[target]
    hist = features.loc[features['Target'].eq(target)].sort_values('Fecha_Date')
    train = hist.loc[hist['Fecha'].lt('2025-01')]
    valid = hist.loc[hist['Fecha'].ge('2025-01')]
    pred = predictions.loc[predictions['Target'].eq(target) & predictions['Model'].eq(model)].sort_values('Fecha_Date')
    fc = forecasts.loc[forecasts['Target'].eq(target) & forecasts['Model'].eq(model)].sort_values('Fecha_Date')
    metric = final_metrics.loc[final_metrics['Target'].eq(target)].iloc[0]

    if pred.empty:
        raise ValueError(f'Missing 2025 prediction rows for {target} / {model}')
    if fc.empty:
        raise ValueError(f'Missing 2026-2027 forecast rows for {target} / {model}')

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(train['Fecha_Date'], train['Consumo_Tm'], color='#333333', linewidth=2.2, label='Training actuals 2023-2024')
    ax.plot(valid['Fecha_Date'], valid['Consumo_Tm'], color='#777777', linewidth=2.2, label='2025 validation actuals')
    ax.plot(pred['Fecha_Date'], pred['Pred'], color='#D95F02', linewidth=2.2, linestyle='--', label=f'2025 selected prediction ({model})')
    ax.plot(fc['Fecha_Date'], fc['Forecast'], color=TARGET_COLORS[target], linewidth=2.4, label=f'2026-2027 selected forecast ({model})')
    ax.axvline(pd.Timestamp('2025-01-01'), color='#999999', linestyle=':', linewidth=1.3)
    ax.axvline(pd.Timestamp('2026-01-01'), color='#999999', linestyle=':', linewidth=1.3)

    if model == 'SARIMA':
        ci = sarima_ci.loc[sarima_ci['Target'].eq(target)].copy()
        ci['Fecha_Date'] = pd.to_datetime(ci['Fecha'])
        ci = ci.sort_values('Fecha_Date')
        ax.fill_between(
            ci['Fecha_Date'], ci['CI_Lower'], ci['CI_Upper'],
            color=TARGET_COLORS[target], alpha=0.15,
            label=f'{ci_level_pct}% prediction interval (calibrated)',
        )
    else:
        err = selected_error.get(target, {'MAPE': 20.0, 'RMSE': 0.0})
        pct_width = fc['Forecast'] * (float(err['MAPE']) / 100.0)
        abs_width = pd.Series(float(err['RMSE']), index=fc.index)
        band_width = np.maximum(pct_width.values, abs_width.values)
        ax.fill_between(
            fc['Fecha_Date'],
            np.maximum(fc['Forecast'].values - band_width, 0),
            fc['Forecast'].values + band_width,
            color=TARGET_COLORS[target], alpha=0.12,
            label='illustrative error band (not calibrated)',
        )

    ax.set_title(f'{target} — selected model: {model} | 2025 MAPE: {metric.MAPE:.1f}% | R2: {metric.R2:.3f}', fontweight='bold')
    ax.set_xlabel('Month')
    ax.set_ylabel('Biodiesel demand (metric tonnes)')
    ax.grid(True, alpha=0.25)
    ax.legend(loc='upper left', fontsize=8)
    fig.tight_layout()

    out = FIGS / f'12_business_{safe_target_name(target)}_train_validation_forecast.png'
    fig.savefig(out, dpi=150, bbox_inches='tight')
    plt.close(fig)
    figure_rows.append({'Target': target, 'Model': model, 'Figure': str(out.relative_to(REPO_ROOT))})

figure_manifest = pd.DataFrame(figure_rows)
display(figure_manifest)

#### Nacional

![Nacional training, validation, and forecast](../reports/figures/12_business_national_train_validation_forecast.png)

Training climbs from near zero through most of 2023 to 13,441 Tm/month by December 2024. The Logistic prediction for 2025 extrapolates that late-training acceleration almost linearly, rising from 14,458 to 31,268 Tm by December -- but actual demand peaks earlier, at 27,001 Tm mid-year, then pulls back to end the year at 22,632 Tm, well below the prediction. The 2026-2027 forecast resumes from that lower actual level (23,286 Tm in January 2026) and stays close to flat, rising only to a peak of 28,027 Tm before easing to 26,092 Tm by December 2027 -- consistent with a curve whose fitted ceiling is now barely above the highest demand it has already observed (Section 4.3). The illustrative band (not calibrated) widens only moderately and stays roughly proportional to the forecast level throughout, since it is a fixed multiple of the 2025 holdout error, not a property of the model's own fitted uncertainty.

#### Madrid

![Madrid training, validation, and forecast](../reports/figures/12_business_madrid_train_validation_forecast.png)

Training is the most lopsided of the five: demand sits near zero through almost all of 2023 and early 2024 before a sharp late takeoff to 2,067 Tm by December 2024. The Logistic prediction for 2025 continues that steep late-training slope almost in a straight line, from 2,584 up to 5,785 Tm -- but actual demand never gets close, peaking at 3,854 Tm mid-year and ending 2025 at just 2,975 Tm. This is the largest gap between prediction and validation of any of the five targets (73.6% MAPE, R2 of -8.273, the weakest of all five by both measures), and it is visible directly as the widest separation between the dashed orange line and the grey actuals on the chart. The 2026-2027 forecast starts much closer to that more recent, lower level (3,486 Tm) and stays nearly flat through to 3,729 Tm by December 2027; the illustrative band is already wide in the very first forecast month, reflecting how large the 2025 error already was.

#### Cataluña

![Cataluña training, validation, and forecast](../reports/figures/12_business_cataluna_train_validation_forecast.png)

Training shows the same shape as the other regions (flat near zero through 2023, a real climb to 2,176 Tm by December 2024), but the SARIMA prediction for 2025 goes the wrong way: it starts close to the actual level (2,286 Tm) and then declines to just 800 Tm by December, while real demand instead climbs and oscillates between 1,882 and 3,811 Tm, ending the year at 3,362 Tm. SARIMA extrapolated the most recent month-over-month changes it had seen in training -- which happened to be a deceleration -- rather than the underlying multi-year growth trend, the opposite error direction from the curve-fit targets above, which overshoot instead of undershoot. The 2026-2027 forecast essentially freezes at the last actual level (3,338 to 3,398 Tm across the full 24 months) rather than resuming growth, and the chart's calibrated 50% interval is the most visually dramatic of the five: a normal-looking +/-19% range in January 2026 (2,689-4,143 Tm) that widens to 793-14,548 Tm by December 2027, almost entirely on the upside. That asymmetric explosion is a real, mathematically honest property of this model's own fitted uncertainty, not an illustrative guess -- and it is shown here at the 50% level specifically because the textbook 95% level is even wider and effectively unreadable (see this section's intro above).

#### Andalucía

![Andalucía training, validation, and forecast](../reports/figures/12_business_andalucia_train_validation_forecast.png)

Andalucía has the smallest, choppiest training history of the five -- it is also the target with the isolated, investigated-and-explained single-month outlier in April 2023 (see `memory.md`) -- ending training at only 684 Tm/month. As with Cataluña, the SARIMA prediction for 2025 stays almost flat (684 down to 594 Tm), while actual demand surges far above that, peaking at 2,664 Tm mid-year and settling at 1,911 Tm by December -- proportionally the largest miss of the five regions on this chart, even though Andalucía's R2 (-1.929) looks less extreme than Madrid's or Cataluña's simply because the absolute scale is smaller. The 2026-2027 forecast picks up from that 2025 ending level (2,050 Tm in January 2026) and holds almost perfectly flat through 2,088 Tm by December 2027. Like Cataluña, the calibrated 50% band explodes asymmetrically with the horizon, from 1,711-2,456 Tm in the first forecast month to 417-10,445 Tm by the last -- again the model's own honest uncertainty, not an illustrative proxy.

#### Valencia

![Valencia training, validation, and forecast](../reports/figures/12_business_valencia_train_validation_forecast.png)

Valencia shows the most gradual, lowest-amplitude training ramp of the five, only beginning a visible climb in 2024 and reaching 549 Tm by December. The Gompertz prediction for 2025 stays close to flat (527 to 570 Tm) while actual demand spikes well above it -- peaking at 1,448 Tm mid-year, dropping back, then climbing to end the year at 1,273 Tm -- the same underestimate pattern seen in Nacional's and Madrid's curve-fit models, just smaller in absolute terms. What sets Valencia apart is the 2026-2027 forecast itself: unlike the other four targets, which all flatten out near their most recent actual level, Valencia's forecast keeps genuinely climbing, from 1,212 Tm in January 2026 to a peak of 1,803 Tm in mid-2027 before easing slightly to 1,732 Tm by December. This matches the driver analysis in Section 4.3: Valencia's fitted Gompertz ceiling sits about 29% above the observed historical maximum, meaning the model itself believes there is still real headroom for adoption to grow -- unlike Nacional's and Madrid's curves, which are now fitted almost exactly at the ceiling they have already observed. The illustrative band widens moderately in step with the rising forecast, since it scales with the 2025 holdout error rather than ballooning with horizon the way the two SARIMA targets' calibrated bands do.

### 1.5 Existing Summary Figures from Earlier Notebooks

These are the existing production figures created by the previous notebooks / script outputs. They are included here for continuity with the modeling work already completed.

![Train/test split](../reports/figures/10_train_test_split.png)

![Actual vs predicted validation results](../reports/figures/11_predictions_vs_actuals.png)

![Selected 24-month forecast](../reports/figures/11_forecast_24m.png)



## 2. Model-by-Model Limitations

| Model family | Where it appears in this project | Main limitations for this project |
|---|---|---|
| SARIMA | Baseline diagnostic only | Uses target history and seasonality, but does not directly use macro, diesel-market, mandate, price, or Repsol-specific business variables. It can still beat candidate models for Nacional, which is an important benchmark caveat. |
| SARIMAX | Headline candidate, but never selected for any target | Adds engineered regressors to SARIMA, but with only ~21-34 usable rows per target and 8 exogenous regressors, the single fit used for the 2025 holdout evaluation collapses (fails to converge, or residual variance falls to near zero) for all 5 targets, not just some of them, so it never produces a usable 2025 holdout metric anywhere (see `data/outputs/degenerate_fits.csv`). In the training-only walk-forward selection that actually picks the headline model, it does produce a real (non-degenerate) score for 4 of 5 targets (every fold fails for Cataluña specifically), but it never wins: its best showing is Andalucía at 90.6% walk-forward MAPE against the winning SARIMA's 48.8%. |
| Logistic / Gompertz growth curves | Headline-eligible adoption-curve candidates | Useful adoption-shape benchmarks, but they do not directly use engineered explanatory variables. |
| Ridge | Headline candidate | Sensitive to collinearity among lag, trend, rolling, macro, CNMC, and mandate features. It can extrapolate unstable trends when used recursively over many months. |
| Random Forest | Headline candidate, not currently selected for any target | Can capture nonlinear patterns, but tends to interpolate within learned ranges and can flatten forecasts. With a very small monthly dataset, feature importance and validation performance can be unstable. |
| XGBoost | Headline candidate, not currently selected for any target | Flexible but data-hungry. With limited history, it can overfit candidate patterns or fail to extrapolate structural adoption behavior. |
| Diesel Share | Headline candidate | Depends on diesel-market proxy assumptions and future diesel-market extrapolation. It is useful as a business-logic benchmark, but validated poorly here. |
| Pooled regional ML | Diagnostic sensitivity family only | Beats the production model on the 2025 holdout for all 4 regional targets (Section 4.2), but is excluded by design, not for underperforming: this project's standing principle is one independently-fit model per region, and pooling's fit is shaped by other regions' data by construction. Retained only as a disclosed sensitivity comparison. |

## 3. Why Model Performance Differs by Region

The exact candidate rankings are shown in the production walk-forward table. The interpretation below explains the selected model per target without replacing the metric tables.

| Target | Final selected model | Interpretation |
|---|---|---|
| Nacional | Logistic | The national series is well described by a saturating adoption curve in the training-only comparison. The mandate and CNMC features were evaluated through SARIMAX/ML, but not forced into the headline model, and SARIMAX itself was excluded here as a degenerate fit. |
| Madrid | Logistic | Madrid is best described by a saturating adoption curve in the training-only walk-forward gate, but the 2025 holdout remains weak. |
| Cataluña | SARIMA | Cataluña's SARIMAX fit was found to be numerically degenerate (non-convergent, near-zero residual variance, the same collapse every target's SARIMAX fit shows at this evaluation stage) and is excluded. Plain SARIMA, a fit that actually converges, wins the training-only comparison; its 2025 holdout MAPE (50.1%) is the third-weakest of the five targets by that metric, though by R2 (-7.182) it is the second-weakest, behind only Madrid, so it remains a risk case worth disclosing, just no longer driven by a broken fit. |
| Andalucía | SARIMA | SARIMA wins after the degeneracy guard rejects flat and non-convergent orders (checked on both the training window and the full history). The selected path is not constant, but it is still fairly flat (24-month range of ~158 Tm) and uncertainty remains high. |
| Valencia | Gompertz | Valencia remains best captured by a saturating growth curve; engineered-variable candidates were evaluated but did not win. |


## 4. Feature and Driver Interpretation

Engineered variables are evaluated through SARIMAX and the direct ML families, but the selection process does not force those models to win. Feature diagnostics should therefore be interpreted as candidate-model diagnostics, not as causal proof and not necessarily as drivers of every selected headline forecast.

| Model type | What drives the model in this project | What should not be overclaimed |
|---|---|---|
| SARIMAX | Target-history dynamics plus seasonal, macro, CNMC diesel-market, and mandate regressors. | With this little history, exogenous coefficient estimates are fragile enough that the single fit used for the 2025 holdout evaluation fails to converge for all 5 targets, not just some of them; SARIMAX is excluded from the headline forecast everywhere, and even in the training-only walk-forward selection, where it does produce a real score for most targets, it never wins. |
| Direct ML | Lagged target, rolling target features, lagged macro indicators, lagged CNMC diesel-market features, and mandate variables. | Validation performance and feature importance are model diagnostics, not causal estimates. |
| SARIMA / Logistic / Gompertz | History, trend, seasonality, and adoption-shape assumptions. | They can be valid headline winners, but they do not directly move with mandate or macro variables. |
| Pooled ML / Diesel Share | Sensitivity diagnostics only. | They are not headline-eligible under the current independent per-target production design. |

The existing feature-importance figure from notebook 09 is included below as a model diagnostic, not as proof of causal drivers.


### 4.1 Average Feature Importance (Random Forest / XGBoost Diagnostic Candidates)

The figure below is the feature-importance diagnostic from `notebooks/09_evaluation.ipynb`: Random Forest mean-decrease-impurity and XGBoost gain, averaged across all 5 targets. It is the only feature-importance artifact this project produces, and it is shown here in full for transparency.

**Read it together with Section 4.2 below: Random Forest and XGBoost are not the selected model for any of the 5 regions.** This chart shows which inputs the losing candidates leaned on, not which inputs drive the forecast Repsol is actually being given. `Lag_1` (last month's demand) dominates both models by a wide margin. That is expected, not a sign of a richer signal being found: with only about 24 monthly training rows per target, a tree-based model has little statistical room to separate the contribution of slower-moving inputs (macro indicators, the mandate schedule, lagged diesel-market features) from the dominant month-to-month carry-over, so importance concentrates almost entirely on the single most recent observation.

![Average feature importance, Random Forest and XGBoost](../reports/figures/10_feature_importance.png)


### 4.2 Why No Tree-Based Model Was Selected for Any Region

Random Forest and XGBoost are two of the seven headline-eligible candidates for every target, but the training-only walk-forward gate (Section 1.3, `model_selection_walkforward.csv`) never picked either one, for any region, and not by a narrow margin. The table generated below shows the winning model's walk-forward MAPE next to Random Forest's and XGBoost's: in every region both tree-based candidates score tens of percentage points of MAPE worse than the winner. This is a structural pattern across all five targets, not a near-miss in one or two of them.

**Why this happens, in order of importance:**

1. **Not enough rows to learn from, relative to the number of inputs.** Each target has only 24 training months (2023-2024), and the ML feature set (`ML_FEATS` in `scripts/05_modeling_with_cnmc.py`) has 17 columns: lags, rolling means, calendar terms, and lagged macro, diesel-market, and mandate variables. A tree ensemble choosing splits across 17 candidate variables with that few rows has very little data behind any one split, so it tends to fit the idiosyncrasies of 2023-2024 rather than a pattern that generalizes to 2025. This is the same root constraint that makes SARIMAX numerically degenerate for every target's 2025 holdout fit (Section 1.3): too many free parameters for too few rows. It just shows up differently for a tree ensemble (unstable, high-variance splits and weak validation scores) than for a likelihood-based model (an outright convergence failure).
2. **Trees cannot extrapolate past the range of values they were trained on, and biodiesel demand is still on a rising adoption curve.** A regression tree predicts the average target value of whichever training rows land in the same leaf as a new input. Once a recursive, multi-month forecast pushes `Lag_1` (last month's demand) above anything the model ever saw in training, which is exactly what happens while demand keeps growing, the model cannot output a value higher than that leaf has ever produced. Over a 12-month-plus recursive horizon this means Random Forest and XGBoost structurally tend to flatten out instead of continuing the trend, which is the same failure mode already named in Section 2's limitations table ("can flatten forecasts").
3. **This is not a careless-defaults problem.** `scripts/05_modeling_with_cnmc.py` already constrains both models specifically to fight overfitting at this sample size (Random Forest: `max_depth=3, min_samples_leaf=3`; XGBoost: `max_depth=2, learning_rate=0.05, reg_alpha=1, reg_lambda=5`). The gap above persists even with that regularization already applied.
4. **The diagnostic pooled-model comparison confirms this is a sample-size effect, not a verdict against tree models for this problem.** As a sensitivity check (never eligible for the headline forecast), the pipeline also fits a *pooled* version of the same Random Forest / XGBoost that stacks all 4 regional targets together (Madrid, Cataluña, Andalucía, Valencia; Nacional is not part of the regional pool), roughly four times the effective rows of any single-region fit. The table generated below (`phase2_pooling_decision.csv`) shows that pooled version beating the current production model on the 2025 holdout for all four regional targets, by a wide margin for three of them (Madrid 33.3% vs. 73.6%, Cataluña 36.4% vs. 50.1%, Andalucía 20.9% vs. 52.6%) and by a narrower but still real margin for the fourth (Valencia 31.8% vs. 34.2%). The same algorithm, given more rows, performs much better. That is direct evidence the limitation is the amount of data available per region today, not something inherent to tree-based methods for biodiesel demand.
5. **Pooling has a real, but resolved, history worth disclosing.** An earlier version of this pooled diagnostic, run at the same shallow tree depth used for the per-region models, genuinely could not tell the four regions apart: it produced near-identical forecasts across regions despite the region-indicator columns being present, because a shallow tree has too few splits to spend on a categorical region flag once the numerically richer lag and rolling-mean features dominate. This was diagnosed and fixed by giving the pooled variant more capacity, confirmed by an explicit check that different regions now produce different forecast paths; the result shown above reflects that fixed version. Separately, an earlier pooled Ridge experiment failed for an unrelated reason: as a linear model with no saturation ceiling, it extrapolated explosively over the 12-month horizon, which is why Ridge does not appear in the pooled comparison above.
6. **Even with point 4's evidence, pooling is excluded by design, not because it lost a fair fight.** This project's standing modeling philosophy is one model per region: every target is free to be won by any model family at all -- the seven-candidate grid in Section 1.3 already searches across statistical and curve-fit families as well as ML for each target independently -- but the winning model's parameters must come entirely from that region's own history. A model whose splits or coefficients are also shaped by another region's rows, however useful that extra data might be, is a different kind of object than "the best model for this region," which is the actual design goal here. Pooling is excluded for that reason, not because point 4's holdout comparison was somehow unfair to it -- that comparison is real and disclosed precisely so this tradeoff is visible, not hidden.

In [ ]:
display(Markdown('#### Training-only walk-forward MAPE: winning model vs. tree-based candidates'))
tree_gap = walkforward[
    ['Target', 'Selected_Model', 'Selected_Model_Training_WalkForward_MAPE', 'Random Forest', 'XGBoost']
].rename(columns={
    'Selected_Model': 'Winning model',
    'Selected_Model_Training_WalkForward_MAPE': 'Winning WF-MAPE (%)',
    'Random Forest': 'Random Forest WF-MAPE (%)',
    'XGBoost': 'XGBoost WF-MAPE (%)',
}).round(1)
display(tree_gap.sort_values('Target').reset_index(drop=True))

display(Markdown('#### Diagnostic: same Random Forest / XGBoost family, pooled across regions (2025 holdout)'))
display(
    pooling[
        ['Target', 'Production_Model', 'Production_MAPE', 'Best_Pooled_Model', 'Best_Pooled_MAPE', 'Best_Pooled_Beats_Production']
    ].sort_values('Target').reset_index(drop=True)
)


### 4.3 What Actually Drives Each Selected Model

None of the five selected models are tree-based or Ridge-based, so "feature importance" in the usual machine-learning sense does not apply to any of them. The two model families that were actually selected work very differently from each other, and from the ML candidates above:

- **SARIMA (Cataluña, Andalucía)** forecasts each month from a small number of *lag terms*: how much demand changed last month and the month before (the non-seasonal part of the order), and how it compared to the same month one year ago (the seasonal part). It is the only one of the five selected models that looks at recent actual values at all.
- **Logistic (Nacional, Madrid) and Gompertz (Valencia)** are saturating growth curves fit directly against elapsed time, plus a fixed monthly seasonal correction. They have no lag terms whatsoever: once fitted, a forecast for any future month depends only on how many months that point is from the start of the series and which calendar month it falls in. They do not look at, or react to, the most recent actual demand figure at all.

The two tables below were generated by `scripts/07_selected_model_drivers.py`, which refits each region's already-selected model on the full 2023-2025 history (the same refit `scripts/05_modeling_with_cnmc.py` already performs internally to produce the 2026-2027 forecast itself) and saves the fitted coefficients, since the production pipeline computes this detail but does not otherwise write it to any file.


In [ ]:
display(Markdown('#### SARIMA lag and seasonal terms, fit on the full 2023-2025 history (Cataluña, Andalucía)'))
display(
    sarima_drivers[
        ['Target', 'Production_Order', 'Production_Seasonal_Order', 'Term', 'Meaning', 'Coefficient', 'Std_Err', 'P_Value', 'Significant_5pct']
    ].sort_values(['Target', 'Term']).reset_index(drop=True)
)


**Cataluña** is fit as `(p,d,q)(P,D,Q,m) = (0,1,2)(1,0,0,12)`. The `d=1` means the model works on month-over-month *changes* in demand rather than the raw level, on the assumption that the level itself trends rather than sitting still. With `p=0`, there is no autoregressive term at all, so the model does not look directly at last month's change; instead, `q=2` gives it two moving-average terms, meaning this month's forecast is corrected by the size of the forecast errors ("shocks") from the previous two months. `P=1` adds one seasonal autoregressive term at lag 12, comparing this month's change to the change seen in the same month one year ago.

This is also the one target where the SARIMA order required the disclosed safety override from Section 1.3: the order with the single best training-only walk-forward score, (0,1,1)(1,0,0,12) at 63.7% MAPE, ships a near-flat 2026-2027 forecast once refit on the full 2023-2025 history, so the next-best training-only candidate shown here, (0,1,2)(1,0,0,12) at 66.9% MAPE, is used instead. This is the same order Cataluña was already shipping before that disclosure existed; nothing about the forecast changed, only the documentation of why this specific order was chosen over the nominally-better-scoring one.

**Andalucía** is fit as `(1,1,2)(1,0,0,12)`: the same seasonal structure, but with `p=1` (one autoregressive term added on top of the two moving-average terms), so last month's change directly informs this month's forecast as well as the two prior forecast errors.

**The honest caveat the table above makes visible: none of these individual lag or seasonal terms are statistically significant at the 5% level, for either region.** Every `P_Value` in the `ar.*`/`ma.*`/`ar.S.*` rows is well above 0.05 (Cataluña's smallest is 0.337; Andalucía's smallest is 0.089). Only `sigma2`, the residual variance, is significant, and that only confirms there is genuine unexplained noise in the series, not that the specific AR/MA/seasonal structure chosen is a real, identified signal. In plain terms: this is the order that scored best among a small, training-only candidate grid (Section 1.3), but with only 36 monthly observations per region, the data cannot pin down *which* lag pattern is responsible with any statistical confidence. Treat the specific order as "the most defensible choice available," not as a discovered causal mechanism, the same caution the project already applies to the Ljung-Box residual diagnostics in `notebooks/09_evaluation.ipynb`.

In [ ]:
display(Markdown('#### Growth-curve shape parameters, fit on the full 2023-2025 history (Nacional, Madrid, Valencia)'))
display(
    curve_params[['Target', 'Curve_Type', 'Parameter', 'Meaning', 'Value']]
    .sort_values(['Target', 'Parameter']).reset_index(drop=True)
)

display(Markdown('#### Seasonal correction on top of the curve (same three regions)'))
display(
    curve_seasonal[
        ['Target', 'Curve_Type', 'Sin_Coef', 'Sin_P_Value', 'Cos_Coef', 'Cos_P_Value',
         'Any_Seasonal_Term_Significant_5pct', 'Seasonal_R2', 'Amplitude_Tm', 'Peak_Month', 'Trough_Month']
    ].sort_values('Target').reset_index(drop=True)
)


**Nacional and Madrid (Logistic)** both estimate an asymptote (`L`) only about 1% above the highest demand actually observed in 2023-2025, with an inflection point (the steepest part of the growth curve) at month-index 26 for Nacional and 26 for Madrid, roughly February 2025, near the end of the observed history. In plain terms, both curves believe the steepest phase of adoption growth has already happened and that 2026-2027 sits on the flattening part of the S-curve, not on continued rapid growth. That is a structural assumption coming from the choice of model family, not a new pattern independently re-derived from data the curve has not already seen. Nacional's seasonal correction (July peak, January trough) has a statistically significant cosine term (p=0.035) but explains only 14% of the residual variance; Madrid's seasonal wiggle (June peak, December trough) is not statistically distinguishable from noise at all (sin p=0.87, cos p=0.17), so it should not be read as a real calendar effect at this sample size.

**Valencia (Gompertz)** tells a different story: its asymptote is about 29% above the observed historical maximum, meaning the model believes there is still real headroom for demand to grow, not flattening yet. Its July peak / January trough seasonal correction is statistically significant (cosine p=0.027), similar in shape to Nacional's.

**The structural takeaway for Repsol:** once fitted, none of Nacional's, Madrid's, or Valencia's forecasts look at recent actual demand at all. If real 2026 demand comes in above or below what the curve assumed, these three models will not adjust until the pipeline is re-run on new data, an argument for the re-run cadence already recommended in Section 6. SARIMA (Cataluña, Andalucía) is the only family among the five that does react to recent actuals each month, but per Section 4.3 above, its own lag terms are not individually statistically significant either, so neither family should be presented to Repsol as a confidently identified causal mechanism. Both should be presented as the best-performing structure found at this sample size, which is a materially different and more honest claim.


### 4.4 Cross-Check Against the Mini Trend Model (notebook 11)

`notebooks/11_mini_trend_regulation_model.ipynb` is a separate, deliberately simple one-slide model that fits its own Logistic/Gompertz curve per target (choosing whichever has the higher in-sample R2, no walk-forward validation) and saves three illustrative scenarios (Conservative / Central / Optimistic, a 0.5x/1.0x/1.5x pace factor on the fitted growth) to `data/outputs/mini_model_scenarios.csv`. It is intentionally not reconciled with the production forecast. The table below checks how far apart the two actually are, and why.

In [ ]:
display(Markdown('#### Production forecast vs. mini-model scenarios, 2026-2027 totals (Tm), and which curve each method picked'))
display(
    mini_check[
        ['Target', 'Production_Model', 'Production_24m_Total_Tm',
         'Mini_Conservative_24m_Total_Tm', 'Mini_Central_24m_Total_Tm', 'Mini_Optimistic_24m_Total_Tm',
         'Mini_Chosen_Curve', 'Mini_InSample_R2_Logistic', 'Mini_InSample_R2_Gompertz',
         'Same_Curve_Family_As_Production']
    ].sort_values('Target').reset_index(drop=True)
)


**Valencia is the only target where production and the mini model genuinely agree.** Both independently choose Gompertz (production via training-only walk-forward; the mini model via in-sample R2: 0.940 vs. Logistic's 0.938), and since both fit the identical Gompertz formula on the identical `Consumo_Tm` history, the result is byte-identical: production's total equals the mini model's Central scenario exactly. This is a real methodological agreement, not a coincidence.

**Nacional and Madrid look close, but for the wrong reason.** Production's selected model is Logistic for both, but the mini model's in-sample R2 favors Gompertz for both too (Nacional: 0.956 vs. 0.954; Madrid: 0.947 vs. 0.937) — `Same_Curve_Family_As_Production` is `False` for both. The two methods are not agreeing; they are fitting two different curve families that happen to produce similar-shaped 24-month forecasts for these two series (within 0.1-0.3%). That is itself a useful, slightly concerning finding: it shows Logistic and Gompertz are barely distinguishable on this data (R2 differs by 0.001-0.01), which is the same curve-identifiability risk already raised in Section 4.3 — with only 36 monthly points, the data cannot confidently tell two structurally different saturating-curve hypotheses apart, so "which curve wins" is not a strongly settled question for either method.

**Cataluña and Andalucía diverge for a structural reason: the mini model cannot produce a SARIMA-shaped forecast at all.** It only ever fits a Logistic or Gompertz curve, so for the two targets where the validated, training-only walk-forward process determined a lag-based SARIMA model fits the actual data better than any smooth growth curve, the mini model's curve-only framework cannot replicate that shape. The gap is real (12% for Cataluña, 25% for Andalucía between production and the mini model's Central case), and notably, production's number sits **below even the mini model's "Conservative" scenario** for both — the validated forecast is more cautious than the narrative model's deliberately low case, not less.

**Takeaway:** the mini model is a useful sense-check, not a second opinion that independently confirms the production forecast. It agrees by construction where both methods land on the same curve family, drifts by coincidence where they land on different but similarly-shaped curves, and diverges structurally wherever the validated pipeline selected a non-curve model.


## 5. What Non-Public Internal Repsol Data Would Improve the Forecasts

The current model predicts market demand. To convert this into stronger Repsol-specific planning, the most valuable additions would be internal operational and commercial data.

| Internal data category | Examples | Why it would improve the model |
|---|---|---|
| Repsol sales volumes | Monthly or weekly sales by station, product, region, and customer segment. | Separates total market demand from Repsol-specific demand and market share. |
| Station-level demand | Station coordinates, catchment area, local traffic, product availability, and station format. | Allows local demand modeling instead of only CCAA-level market forecasting. |
| Repsol pricing and promotions | Pump prices, discounts, loyalty offers, B2B contract prices, campaign dates. | Enables price-response and promotion-response modeling. |
| Product availability | Biodiesel/HVO availability by station and date, rollout dates, stockout records. | Distinguishes true demand from constrained sales caused by supply or availability. |
| Inventory and logistics | Tank capacity, replenishment dates, delivery lead times, stock levels, distribution constraints. | Connects demand forecasts to operational planning and risk of shortages. |
| Customer behavior | Fleet accounts, loyalty-card behavior, repeat customers, B2B vs retail split. | Improves segmentation and identifies demand sources with different behavior. |
| Competitive context | Nearby competitor stations, competitor prices, local diesel alternatives. | Helps explain regional or station-level substitution patterns. |
| Policy and compliance data | Internal compliance targets, blending strategy, contract obligations. | Connects market demand forecasts to Repsol's regulatory and strategic constraints. |



## 6. Business Recommendations for Repsol

These recommendations are intentionally framed as planning guidance, because the current project does not include Repsol internal sales or margin data.

1. Use the current forecast as a **market-demand scenario**, not as a direct Repsol sales forecast.
2. Treat regional forecasts as **directional signals** and monitor actuals closely, especially where validation metrics are weak.
3. Present the selected forecast (SARIMA/Logistic/Gompertz, one model per region) as the official headline model set, and keep the non-selected candidates (SARIMAX, Ridge, Random Forest, XGBoost, plus the diagnostic-only pooled regional ML and Diesel Share candidates) visible in the appendix as transparent comparison baselines, not hidden alternatives.
4. Build a Repsol-specific demand layer on top of the market forecast using internal sales, pricing, station, availability, and customer data.
5. Use the model outputs to support regional scenario planning, not single-point operational commitments.
6. Re-run the pipeline when newer actual demand data becomes available and track forecast drift by region.
7. For business decisions such as inventory, pricing, and logistics, require internal data validation before acting on model outputs.


## 7. Priority Data Roadmap

| Priority | Data / modeling improvement | Purpose |
|---:|---|---|
| 1 | Add Repsol sales by station, product, customer type, and month. | Convert market-demand forecasting into Repsol-demand forecasting. |
| 2 | Add product availability, inventory, and stockout history. | Separate demand from supply constraints. |
| 3 | Add Repsol pricing, discounts, promotions, and B2B contract terms. | Estimate commercial levers and price sensitivity. |
| 4 | Add customer/fleet segmentation and loyalty behavior. | Improve regional and station-level demand segmentation. |
| 5 | Add competitor/local market context around stations. | Explain local substitution and market-share differences. |
| 6 | Extend the actual demand history when new 2026 data becomes available. | Strengthen backtesting and reduce reliance on a single 2025 validation year. |
| 7 | Develop a two-layer model: market demand first, Repsol share second. | Keep the current capstone model useful while making it operational for Repsol. |


## Final Caution

The most important business message is that the current project is a solid market-demand forecasting framework, but not yet a full Repsol operational forecast. The next step is not simply a more complex algorithm; it is adding Repsol internal data so the model can distinguish market growth from Repsol-specific sales, availability, pricing, customer behavior, and logistics constraints.
